# Reviews Preprocessing Pipeline

Steps:
1. Lowercasing  
2. Remove URLs  
3. Remove HTML tags  
4. Convert emojis → words  
5. Remove punctuation & extra spaces  
6. Tokenization  
7. Stopword removal (keep negations)  
8. Lemmatization  

**Stopword Decision:**  
We remove stopwords but keep negations (e.g., "not", "no") because they affect sentiment.

### Imports & NLTK Setup

In [1]:
import re
import pandas as pd
import nltk
import emoji

# Download required NLTK data (runs once)
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

### Stopwords & Lemmatizer Setup

In [2]:
NEGATIONS = {"not", "no", "never", "nor", "neither", "nothing",
             "nobody", "nowhere", "hardly", "barely", "scarcely"}

STOP_WORDS = set(stopwords.words("english")) - NEGATIONS

lemmatizer = WordNetLemmatizer()

### Cleaning Functions

In [3]:
def lowercase(text):
    return text.lower()

def remove_urls(text):
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"www\.\S+", "", text)
    return text

def remove_html(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"&[a-z]+;", " ", text)
    return text

def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def remove_punctuation(text):
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"'\s|'\s*$", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

def lemmatize(tokens):
    lemmas = [lemmatizer.lemmatize(t, pos="v") for t in tokens]
    lemmas = [lemmatizer.lemmatize(t, pos="n") for t in lemmas]
    return lemmas

### Full Pipeline Function

In [4]:
def preprocess(text):
    s1 = lowercase(text)
    s2 = remove_urls(s1)
    s3 = remove_html(s2)
    s4 = convert_emojis(s3)
    s5 = remove_punctuation(s4)
    s6 = tokenize(s5)
    s7 = remove_stopwords(s6)
    s8 = lemmatize(s7)

    return {
        "cleaned_text": " ".join(s8),
        "tokens": s8,
        "token_count": len(s8),
    }

### Load Dataset

In [5]:
INPUT_FILE = "reviews_dataset.csv"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} reviews")

df.head()

Loaded 30000 reviews


,review_id,source,product_category,review_text,rating,label
0,REV00001,Google Play Store,Games - Battle Royale,It is a wonderfull game but please fix the voi...,2,negative
1,REV00002,Google Play Store,Games - Social Deduction,Great concept. But the amount of people that q...,4,positive
2,REV00003,Google Play Store,Games - Augmented Reality,The game itself keep force closing during and ...,4,positive
3,REV00004,Google Play Store,Games - Shooter,There's currently a bug in the game that makes...,5,positive
4,REV00005,Google Play Store,Games - Strategy,"its stupid, the game gives u the first 10 aren...",2,negative


### Apply Preprocessing



In [6]:
results = df["review_text"].apply(preprocess)
results_df = pd.DataFrame(results.tolist())

df["cleaned_text"] = results_df["cleaned_text"]
df["tokens"] = results_df["tokens"].apply(lambda t: " | ".join(t))
df["token_count"] = results_df["token_count"]

### Reorder Columns

In [7]:
cols = ["review_id", "source", "product_category",
        "review_text", "cleaned_text", "tokens",
        "token_count", "rating", "label"]

df = df[cols]
df.head()

,review_id,source,product_category,review_text,cleaned_text,tokens,token_count,rating,label
0,REV00001,Google Play Store,Games - Battle Royale,It is a wonderfull game but please fix the voi...,wonderfull game please fix voice glitch happen...,wonderfull | game | please | fix | voice | gli...,34,2,negative
1,REV00002,Google Play Store,Games - Social Deduction,Great concept. But the amount of people that q...,great concept amount people quit midway game n...,great | concept | amount | people | quit | mid...,51,4,positive
2,REV00003,Google Play Store,Games - Augmented Reality,The game itself keep force closing during and ...,game keep force close battle even catch pokémo...,game | keep | force | close | battle | even | ...,42,4,positive
3,REV00004,Google Play Store,Games - Shooter,There's currently a bug in the game that makes...,'s currently bug game make thing go completely...,'s | currently | bug | game | make | thing | g...,49,5,positive
4,REV00005,Google Play Store,Games - Strategy,"its stupid, the game gives u the first 10 aren...",stupid game give first arena easly second play...,stupid | game | give | first | arena | easly |...,40,2,negative


### Save File

In [8]:
OUTPUT_FILE = "reviews_preprocessed.csv"
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Saved to {OUTPUT_FILE}")

Saved to reviews_preprocessed.csv
